In [ ]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
import csv

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input point feature classes of mission background conditions with * completed * shoreline and raster GIS calculations
lf_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_LF_background_points_p26920"
hf_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"

# Output CSVs of background conditions "calc_" point feature classes Attribute Table data
lf_calc_csv = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\calc_LF_background_points_p26920.csv"
hf_calc_csv = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\calc_HF_background_points_p26920.csv"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── INSPECT DATA IN EACH ATTRIBUTE TABLE ─────────────────────────────────────────────────────────────

# Function to inspect structure/data in each point feature class Attribute Table
def inspect_fc_fields(fc_list):
    # loop through each feature class
    for fc in fc_list:
        # print message header identifying which feature class
        print(f"\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ {os.path.basename(fc)} ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
        # loop through every field in each Attribute Table
        for f in arcpy.ListFields(fc):
            # print name, data type, and length of every field
            print(f"🛈 [Field] {f.name}     ⸺ [Data Type] {f.type}      ⸺ [Max bytes/chars] Length={f.length}")


# Call function for "calc_" point feature classes
inspect_fc_fields(lf_points_p26920)
inspect_fc_fields(hf_points_p26920)

In [ ]:
# ── SET DESIRED ORDER OF COLUMNS IN OUTPUT CSV FILES ─────────────────────────────────────────────────

field_order_background = [
    "mission_name",
    "glider_nc_file",
    "glider_profile_id",
    "start_datetime_UTC",
    "end_datetime_UTC",
    "av_datetime_UTC",
    "start_latitude_DD",
    "start_longitude_DD",
    "end_latitude_DD",
    "end_longitude_DD",
    "av_latitude_DD",
    "av_longitude_DD",
    "glider_depth_m",
    "water_temperature_C",
    "sea_surface_water_temperature_C",
    "salinity_PSU",
    "sea_surface_salinity_PSU",
    "oxygen_concentration_μmol_L",
    "sea_surface_oxygen_concentration_μmol_L",
    "bathymetric_depth_m",
    "bathymetric_slope_deg",
    "native_raster_resolution_m",
    "distance_to_shore_m",
    "nearest_shoreline_name",
    "nearest_shoreline_fid",
    "nearest_shoreline_x_coordinate_DD",
    "nearest_shoreline_y_coordinate_DD"
]

In [ ]:
# ── EXPORT EACH ATTRIBUTE TABLE AS A CSV (WITH COLUMNS REORDERED) ────────────────────────────────────

# Function to export a feature class Attribute Table as CSV with fields in order defined above
def export_ordered_csv(fc, out_csv, field_order):
    # get field names present in feature class Attribute Table
    existing = {f.name for f in arcpy.ListFields(fc)}
    # find any fields in 'field_order' list that are NOT present in feature class
    missing = [fname for fname in field_order if fname not in existing]
    # print warning message if any expected fields are missing and * skip export *
    if missing:
        print(f"\n❌[FAILURE] {os.path.basename(fc)} missing field(s): {', '.join(missing)} - export SKIPPED \n")
        return

    # * write CSV directly from SearchCursor so float64 (arcgis "Double" data type) coordinate values retain full precision (csv.writer uses str() on each float64 and emits shortest string that "round-trips" exactly (aka value written to csv is exactly equal to input float64 value))
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        # header row in desired field order
        writer.writerow(field_order)
        # pull each row straight from Attribute Table in specified field order
        with arcpy.da.SearchCursor(fc, field_order) as cursor:
            # loop through each row and write to CSV
            for row in cursor:
                writer.writerow(row)
    # print message confirming CSV was written successfully
    print(f"✅ Exported: {out_csv} \n")


# Call function for "calc_" point feature classes
export_ordered_csv(lf_points_p26920, lf_calc_csv, field_order_background)
export_ordered_csv(hf_points_p26920, hf_calc_csv, field_order_background)